# TabPy Server for Tableau: ML Model Deployment
## Created by: Mohammed Golam Kaisar Hossain Bhuyan
### Date: NOV 29, 2025

### 🔹 Step 1 — Install and Start TabPy

In [ ]:
## From Command Prompt Run Below:
#Command 1: pip install tabpy
#Command 2: tabpy
## Successful run will show: TabPy server running (http://localhost:9004 by default)

### 🔹 Step 2 — Load Your Model (.pkl) in Python

In [ ]:
import pickle

# Provide the directory path of the saved Model
MODEL_PATH = "model/diabetes-risk-pred-kaisar-v1.1.pkl"

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

model

### 🔹 Step 3 — Connect to TabPy from Notebook

In [2]:
from tabpy.tabpy_tools.client import Client
client = Client("http://localhost:9004/")

### 🔹 Step 4 — Define a Prediction Function for TabPy as the 

In [3]:
def predict_diabetes(a1c, glucose_pp, glucose_fast, family_history,
                     age, activity, bmi, systolic_bp):
    
    import pandas as pd
    import numpy as np
    
    df = pd.DataFrame({
        "hba1c": a1c,
        "glucose_postprandial": glucose_pp,
        "glucose_fasting": glucose_fast,
        "family_history_diabetes": family_history,
        "age": age,
        "physical_activity_minutes_per_week": activity,
        "bmi": bmi,
        "systolic_bp": systolic_bp
    })

    # Normalize family history
    df["family_history_diabetes"] = (
        df["family_history_diabetes"]
        .replace({"Yes":1,"No":0,"yes":1,"no":0,True:1,False:0})
        .fillna(0)
        .astype(int)
    )
    
    # Fill missing numeric values
    df = df.fillna(df.median(numeric_only=True))

    # Predict
    probabilities = model.predict_proba(df)[:,1]
    predicted_class = ["Yes" if p >= 0.5 else "No" for p in probabilities]

    return {
        "preds": predicted_class,
        "probs": probabilities.tolist()
    }


### 🔹 Step 5 — Deploy Function to TabPy

In [4]:
# Deploy to TabPy (name in Tableau as "predict_diabetes")
client.deploy('predict_diabetes',
              predict_diabetes,
              'Predicts diabetes Yes/No and probability (probs)')
print("Deployed predict_diabetes to TabPy")

Deployed predict_diabetes to TabPy


### 🔹 Step 6 — Test Call Locally from Python

In [5]:
client.query(
    'predict_diabetes',
    [2.8], [120], [100], ["Yes"], [45], [40], [27.6], [138]
)

{'response': {'preds': ['No'], 'probs': [0.35]},
 'version': 1,
 'model': 'predict_diabetes',
 'uuid': 'eebf62dd-9862-494b-af22-cce733d78988'}

## 🔹 Tableau Clalculated Fields

### 🔹 Tableau Clalculated Field: Predicted Class (Individual)

SCRIPT_STR(
"
result = tabpy.query(
    'predict_diabetes',
    _arg1, _arg2, _arg3, _arg4,
    _arg5, _arg6, _arg7, _arg8
)['response']

return result['preds']
",
[Parameter Hba1C],
[Parameter Glucose Postprandial],
[Parameter Glucose Fasting],
[Parameter Family History],
[Parameter Age],
[Parameter Physical Activity Min per Week],
[Parameter BMI],
[Parameter Systolic BP]
)

### 🔹 Tableau Clalculated Field: Predicted Probability (Individual)

SCRIPT_REAL(
"
result = tabpy.query(
    'predict_diabetes',
    _arg1, _arg2, _arg3, _arg4,
    _arg5, _arg6, _arg7, _arg8
)['response']

return result['probs']
",
 ATTR([Parameter Hba1C]),
 ATTR([Parameter Glucose Postprandial]),
 ATTR([Parameter Glucose Fasting]),
 ATTR([Parameter Family History]),
 ATTR([Parameter Age]),
 ATTR([Parameter Physical Activity Min per Week]),
 ATTR([Parameter BMI]),
 ATTR([Parameter Systolic BP])
)

### 🔹 Tableau Configuration -> Manage Analytics Extensions Connection

Hostname: localhost
Port: 9004

## End